# Практика 6. Описова статистика: міри положення і розсіювання

**Студент:** Yurii_Huz  
**Група:** IT-41  
**Варіант:** 1, Київ, регіон Центр

Мета: використати `.describe()`, перевірити його результати ручними обчисленнями та порівняти описову статистику регіонів і міст.

In [ ]:
import numpy as np
import pandas as pd

months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
kyiv_humidity = pd.Series([84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85], index=months, name="вологість")
print(kyiv_humidity)

## Завдання 1. `describe()` на власному варіанті

Для вологості Києва `count` — кількість спостережень, `mean` — середнє арифметичне, `std` — вибіркове стандартне відхилення, `min` і `max` — мінімум і максимум, `25%` і `75%` — перший і третій квартилі, а `50%` — медіана.

In [ ]:
kyiv_description = kyiv_humidity.describe()
print(kyiv_description)

## Завдання 2. Ручне обчислення і перевірка

In [ ]:
range_value = kyiv_humidity.max() - kyiv_humidity.min()
variance_sample = kyiv_humidity.var(ddof=1)
variance_population = kyiv_humidity.var(ddof=0)
std_sample = kyiv_humidity.std(ddof=1)
q1, q3 = kyiv_humidity.quantile([0.25, 0.75])
iqr = q3 - q1
cv = std_sample / kyiv_humidity.mean() * 100

manual = pd.Series({
    "розмах": range_value,
    "дисперсія_ddof_1": variance_sample,
    "дисперсія_ddof_0": variance_population,
    "стандартне_відхилення": std_sample,
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr,
    "CV_%": cv,
})
print(manual)
print()
print("Q1 збігається з describe():", q1 == kyiv_description.loc["25%"])
print("Q3 збігається з describe():", q3 == kyiv_description.loc["75%"])

Дисперсія з `ddof=1` є вибірковою оцінкою і саме вона використовується для `std` у `describe()`. Дисперсія з `ddof=0` описує всю сукупність і трохи менша, бо ділить суму квадратів відхилень на `n`, а не на `n-1`. Коефіцієнт варіації нормує стандартне відхилення на середнє, тому його зручно використовувати для порівняння наборів різного масштабу.

## Завдання 3. Порівняння регіонів

In [ ]:
data = [
    ["Київ", "Центр", 84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85],
    ["Львів", "Захід", 85, 83, 80, 73, 68, 65, 63, 65, 70, 77, 82, 86],
    ["Одеса", "Південь", 82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83],
    ["Харків", "Схід", 83, 80, 75, 68, 62, 60, 58, 60, 66, 73, 79, 84],
    ["Дніпро", "Схід", 82, 79, 74, 67, 61, 59, 57, 59, 65, 72, 78, 83],
    ["Чернігів", "Центр", 85, 83, 79, 71, 66, 63, 61, 63, 69, 76, 82, 86],
    ["Івано-Франківськ", "Захід", 86, 84, 81, 75, 70, 67, 65, 67, 72, 78, 83, 87],
    ["Полтава", "Центр", 84, 81, 76, 69, 63, 61, 59, 61, 67, 74, 80, 85],
    ["Суми", "Схід", 85, 82, 77, 70, 64, 62, 60, 62, 68, 75, 81, 86],
    ["Ужгород", "Захід", 84, 82, 79, 73, 68, 66, 64, 66, 71, 77, 82, 85],
]
columns = ["Місто", "Регіон"] + months
regions = pd.DataFrame(data, columns=columns)
regions["Середня_вологість"] = regions[months].mean(axis=1)
print(regions[["Місто", "Регіон", "Середня_вологість"]])

In [ ]:
center_means = regions[regions["Регіон"] == "Центр"]["Середня_вологість"]
east_means = regions[regions["Регіон"] == "Схід"]["Середня_вологість"]

center_description = center_means.describe()
east_description = east_means.describe()
print("Описова статистика регіону Центр:")
print(center_description)
print()
print("Описова статистика регіону Схід:")
print(east_description)

center_cv = center_means.std() / center_means.mean() * 100
east_cv = east_means.std() / east_means.mean() * 100
print()
print("CV Центр (%):", center_cv)
print("CV Схід (%):", east_cv)

Порівняння виконано булевою фільтрацією, без `groupby()`. У регіоні Центр середня вологість міст близька між собою, тому його `std` і CV невеликі. У регіоні Схід середні значення також близькі, але розкид між Харковом, Дніпром і Сумами потрібно оцінювати одночасно за `std` і CV: CV показує цей розкид у відсотках від середнього регіону.

## Завдання 4. Форма розподілу власних даних

Для Києва середнє близьке до медіани, тому розподіл місячної вологості ближчий до симетричного, хоча літні мінімальні значення та зимові максимальні значення створюють легку асиметрію. За такої невеликої різниці середнє є репрезентативною мірою, а медіана дає корисну стійку перевірку.

## Завдання 5. Порівняння варіативності двох міст

In [ ]:
lviv = regions[regions["Місто"] == "Львів"][months].iloc[0]
lviv_cv = lviv.std() / lviv.mean() * 100
kyiv_cv = kyiv_humidity.std() / kyiv_humidity.mean() * 100

comparison = pd.DataFrame(
    [
        ["Київ", kyiv_humidity.mean(), kyiv_humidity.std(), kyiv_cv],
        ["Львів", lviv.mean(), lviv.std(), lviv_cv],
    ],
    columns=["місто", "середня_вологість", "std", "CV_%"],
)
print(comparison)